# Filter Personal/Belief-Related Conversations from WildChat Intents

## Pipeline Overview

1. **Load intents** from `wildchat_intents.jsonl`
2. **Filter intents** for personal, belief, mental health, life challenges content
3. **Fetch conversations** from WildChat for filtered hashes
4. **LLM classification** to verify and categorize conversations
5. **Prepare for annotation** of belief change taxonomy
6. **Save** filtered dataset for experiments

## Theoretical Grounding

We're looking for conversations where users might exhibit:
- **Cognitive Dissonance Resolution** (Festinger) — updating beliefs after conflicting info
- **Looking-Glass Self Updates** (Cooley) — adjusting based on feedback
- **Goal Shifts** (Cooper) — changing objectives mid-conversation
- **Front-Stage Adjustments** (Goffman) — modifying self-presentation

In [ ]:
import json
import re
from collections import defaultdict, Counter
from pathlib import Path
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
from datetime import datetime

## Configuration

In [ ]:
# === Input files ===
INTENTS_FILE = "wildchat_intents.jsonl"
PREDICTABILITY_FILE = "user_exploitation_stats.jsonl"  # From predictability notebook

# === Output files ===
FILTERED_INTENTS_FILE = "personal_belief_intents.jsonl"
FILTERED_CONVERSATIONS_FILE = "personal_belief_conversations.jsonl"
ANNOTATION_READY_FILE = "belief_change_annotation_ready.jsonl"
PRIORITIZED_FILE = "prioritized_for_annotation.jsonl"  # Ranked by priority

# === Filtering settings ===
MIN_TURNS = 4  # Minimum conversation turns for meaningful analysis

# === Priority thresholds ===
TIER_1_THRESHOLD = 0.7  # Gold candidates (annotate first)
TIER_2_THRESHOLD = 0.4  # Good candidates (annotate second)

## Step 1: Define Intent Filtering Keywords

Based on our theoretical framework, we want conversations about:
- Personal life, relationships, identity
- Beliefs, opinions, values
- Mental health, emotions, challenges
- Life decisions, dilemmas, advice-seeking
- Self-reflection, personal growth

In [ ]:
# === INCLUDE: Intent patterns we WANT ===
# These are patterns to match in the "intent-generic" field

INCLUDE_PATTERNS = {
    # Personal life & relationships
    'personal_life': [
        r'personal',
        r'relationship',
        r'dating',
        r'marriage',
        r'divorce',
        r'family',
        r'friend',
        r'partner',
        r'boyfriend',
        r'girlfriend',
        r'spouse',
        r'parent',
        r'child',
        r'sibling',
    ],
    
    # Mental health & emotions
    'mental_health': [
        r'mental health',
        r'anxiety',
        r'depress',
        r'stress',
        r'emotion',
        r'feeling',
        r'therapy',
        r'counsel',
        r'cope',
        r'overwhelm',
        r'loneli',
        r'grief',
        r'trauma',
        r'self-esteem',
        r'confidence',
        r'insecur',
        r'worry',
        r'fear',
    ],
    
    # Beliefs, opinions, values
    'beliefs_opinions': [
        r'belief',
        r'opinion',
        r'value',
        r'perspective',
        r'viewpoint',
        r'stance',
        r'position on',
        r'think about',
        r'feel about',
        r'view on',
        r'attitude',
        r'moral',
        r'ethic',
        r'principle',
        r'philosophy',
        r'worldview',
        r'religion',
        r'spiritual',
        r'faith',
        r'political',
    ],
    
    # Life decisions & dilemmas
    'life_decisions': [
        r'decision',
        r'dilemma',
        r'choice',
        r'advice',
        r'guidance',
        r'help.*decide',
        r'should.*do',
        r'what to do',
        r'career',
        r'job',
        r'quit',
        r'moving',
        r'relocat',
        r'life change',
        r'crossroad',
        r'uncertain',
        r'confused about',
    ],
    
    # Self-reflection & identity
    'self_reflection': [
        r'self-reflect',
        r'understand.*self',
        r'identity',
        r'who am i',
        r'purpose',
        r'meaning',
        r'self-improvement',
        r'personal growth',
        r'self-discovery',
        r'introspec',
        r'habit',
        r'change.*behavior',
        r'improve.*self',
    ],
    
    # Conflict & challenges
    'challenges': [
        r'conflict',
        r'problem',
        r'challenge',
        r'struggle',
        r'difficult',
        r'issue',
        r'crisis',
        r'trouble',
        r'deal with',
        r'handle',
        r'manage',
        r'overcome',
    ],
    
    # Persuasion & argument (where beliefs might change)
    'persuasion': [
        r'convinc',
        r'persuad',
        r'argument',
        r'debate',
        r'discuss.*opinion',
        r'change.*mind',
        r'disagree',
        r'agree',
    ],
}

# Compile all include patterns
ALL_INCLUDE_PATTERNS = []
for category, patterns in INCLUDE_PATTERNS.items():
    ALL_INCLUDE_PATTERNS.extend([(p, category) for p in patterns])

In [ ]:
# === EXCLUDE: Intent patterns we DON'T want ===
# These indicate technical/academic conversations unlikely to have belief changes

EXCLUDE_PATTERNS = [
    # Technical/coding
    r'code',
    r'programming',
    r'python',
    r'javascript',
    r'algorithm',
    r'software',
    r'debug',
    r'api',
    r'database',
    r'function',
    
    # Math/science (academic)
    r'mathematical',
    r'equation',
    r'formula',
    r'calcul',
    r'theorem',
    r'proof',
    r'chemistry',
    r'physics',
    r'biology',
    r'chemical',
    r'molecule',
    r'reaction',
    r'lewis acid',
    r'convergence',
    r'sequence',
    r'series',
    r'integral',
    r'derivative',
    
    # Academic tasks
    r'essay',
    r'homework',
    r'assignment',
    r'exam',
    r'test question',
    r'thesis',
    r'research paper',
    r'citation',
    
    # Creative writing (fictional)
    r'write.*story',
    r'fiction',
    r'novel',
    r'character.*create',
    r'roleplay',
    
    # Pure information lookup
    r'translate',
    r'definition',
    r'explain.*concept',
    r'summarize',
    r'list of',
    r'history of',
    r'what is a',
]

In [ ]:
def classify_intent(intent_text: str) -> dict:
    """
    Classify an intent string based on keyword patterns.
    
    Returns:
        dict with:
        - is_personal: bool
        - matched_categories: list of matched include categories
        - matched_patterns: list of specific patterns matched
        - excluded_by: pattern that caused exclusion (if any)
        - confidence: rough confidence score
    """
    intent_lower = intent_text.lower()
    
    # Check exclude patterns first
    for pattern in EXCLUDE_PATTERNS:
        if re.search(pattern, intent_lower):
            return {
                'is_personal': False,
                'matched_categories': [],
                'matched_patterns': [],
                'excluded_by': pattern,
                'confidence': 0.0
            }
    
    # Check include patterns
    matched_categories = set()
    matched_patterns = []
    
    for pattern, category in ALL_INCLUDE_PATTERNS:
        if re.search(pattern, intent_lower):
            matched_categories.add(category)
            matched_patterns.append(pattern)
    
    # Calculate confidence based on number of matches
    if len(matched_patterns) == 0:
        confidence = 0.0
    elif len(matched_patterns) == 1:
        confidence = 0.5
    elif len(matched_patterns) == 2:
        confidence = 0.7
    else:
        confidence = min(0.9, 0.5 + len(matched_patterns) * 0.1)
    
    return {
        'is_personal': len(matched_patterns) > 0,
        'matched_categories': list(matched_categories),
        'matched_patterns': matched_patterns,
        'excluded_by': None,
        'confidence': confidence
    }

## Step 2: Load and Filter Intents

In [ ]:
def load_and_filter_intents(filepath: str) -> tuple:
    """
    Load intents from JSONL file and filter for personal/belief-related content.
    
    Returns:
        (filtered_intents, stats)
    """
    filtered = []
    stats = {
        'total': 0,
        'excluded': 0,
        'no_match': 0,
        'matched': 0,
        'category_counts': defaultdict(int),
        'exclude_reasons': defaultdict(int)
    }
    
    with open(filepath, 'r') as f:
        for line in tqdm(f, desc="Filtering intents"):
            if not line.strip():
                continue
            
            entry = json.loads(line)
            stats['total'] += 1
            
            intent_text = entry.get('intent-generic', '')
            classification = classify_intent(intent_text)
            
            if classification['excluded_by']:
                stats['excluded'] += 1
                stats['exclude_reasons'][classification['excluded_by']] += 1
            elif not classification['is_personal']:
                stats['no_match'] += 1
            else:
                stats['matched'] += 1
                for cat in classification['matched_categories']:
                    stats['category_counts'][cat] += 1
                
                filtered.append({
                    'conversation_hash': entry.get('conversation_hash'),
                    'country': entry.get('country'),
                    'intent': intent_text,
                    'categories': classification['matched_categories'],
                    'confidence': classification['confidence']
                })
    
    stats['category_counts'] = dict(stats['category_counts'])
    stats['exclude_reasons'] = dict(stats['exclude_reasons'])
    
    return filtered, stats

In [ ]:
# Run filtering
filtered_intents, filter_stats = load_and_filter_intents(INTENTS_FILE)

print("\n" + "="*60)
print("INTENT FILTERING RESULTS")
print("="*60)
print(f"Total intents:          {filter_stats['total']:,}")
print(f"Excluded (technical):   {filter_stats['excluded']:,}")
print(f"No match (neutral):     {filter_stats['no_match']:,}")
print(f"Matched (personal):     {filter_stats['matched']:,}")
print(f"Hit rate:               {filter_stats['matched']/filter_stats['total']*100:.2f}%")

print("\nCategory breakdown:")
for cat, count in sorted(filter_stats['category_counts'].items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count:,}")

print("\nTop exclude reasons:")
for reason, count in sorted(filter_stats['exclude_reasons'].items(), key=lambda x: -x[1])[:10]:
    print(f"  {reason}: {count:,}")

In [ ]:
# Preview some filtered intents
print("\n" + "="*60)
print("SAMPLE FILTERED INTENTS")
print("="*60)

# Group by category and show examples
by_category = defaultdict(list)
for intent in filtered_intents:
    for cat in intent['categories']:
        by_category[cat].append(intent)

for cat in list(INCLUDE_PATTERNS.keys())[:5]:
    if cat in by_category and by_category[cat]:
        print(f"\n--- {cat.upper()} ---")
        for intent in by_category[cat][:2]:
            print(f"  {intent['intent'][:100]}...")

In [ ]:
# Save filtered intents
with open(FILTERED_INTENTS_FILE, 'w') as f:
    for intent in filtered_intents:
        f.write(json.dumps(intent) + '\n')

print(f"Saved {len(filtered_intents)} filtered intents to {FILTERED_INTENTS_FILE}")

## Step 3: Fetch Conversations from WildChat

In [ ]:
def fetch_conversations(conv_hashes: set, max_convs: int = None) -> dict:
    """
    Fetch conversations from WildChat by conversation_hash.
    """
    print(f"Fetching {len(conv_hashes)} conversations from WildChat...")
    
    dataset = load_dataset("allenai/WildChat-1M", split="train", streaming=True)
    
    conversations = {}
    target_count = len(conv_hashes) if max_convs is None else min(len(conv_hashes), max_convs)
    scanned = 0
    
    for example in tqdm(dataset, desc="Scanning WildChat"):
        conv_hash = example.get("conversation_hash", "")
        
        if conv_hash in conv_hashes:
            conversations[conv_hash] = {
                "conversation_hash": conv_hash,
                "conversation": example.get("conversation", []),
                "model": example.get("model", "unknown"),
                "turn": example.get("turn", 0),
                "language": example.get("language", "unknown")
            }
            
            if len(conversations) >= target_count:
                print(f"\nFound all {len(conversations)} conversations. Stopping.")
                break
        
        scanned += 1
        if scanned % 100000 == 0:
            print(f"  Scanned {scanned:,}, found {len(conversations)}/{target_count}")
    
    print(f"Fetched {len(conversations)} conversations")
    return conversations

In [ ]:
# Get conversation hashes from filtered intents
conv_hashes = {intent['conversation_hash'] for intent in filtered_intents}
print(f"Unique conversation hashes to fetch: {len(conv_hashes)}")

# Optional: limit for testing
MAX_FETCH = None  # Set to e.g., 1000 for testing

# Fetch conversations
conversations = fetch_conversations(conv_hashes, max_convs=MAX_FETCH)

In [ ]:
# Merge intents with conversations
merged_data = []

for intent in filtered_intents:
    conv_hash = intent['conversation_hash']
    if conv_hash in conversations:
        conv = conversations[conv_hash]
        
        # Filter by minimum turns
        num_turns = len(conv['conversation'])
        if num_turns < MIN_TURNS:
            continue
        
        merged_data.append({
            'conversation_hash': conv_hash,
            'intent': intent['intent'],
            'categories': intent['categories'],
            'confidence': intent['confidence'],
            'country': intent['country'],
            'model': conv['model'],
            'num_turns': num_turns,
            'conversation': conv['conversation']
        })

print(f"\nMerged {len(merged_data)} conversations (with min {MIN_TURNS} turns)")

In [ ]:
# Save merged data
with open(FILTERED_CONVERSATIONS_FILE, 'w') as f:
    for item in merged_data:
        f.write(json.dumps(item, default=str) + '\n')

print(f"Saved {len(merged_data)} conversations to {FILTERED_CONVERSATIONS_FILE}")

## Step 3.5: Integrate Predictability Scores & Compute Priority

We prioritize conversations that are:
1. **Unpredictable** (low P₁, low α) — users exploring new ground
2. **Personal** — discussing beliefs, emotions, life decisions

**Theoretical basis**: Belief changes happen during exploration (low exploitation),
not when users repeat patterns. Low P₁/α indicates the user is processing new information,
which is exactly when Cognitive Dissonance occurs.

In [ ]:
def load_predictability_scores(filepath: str) -> dict:
    """
    Load predictability scores from user_exploitation_stats.jsonl.
    
    Returns:
        dict mapping conversation_hash -> {p1, alpha, predictability_type, ...}
    """
    scores = {}
    
    try:
        with open(filepath, 'r') as f:
            for line in f:
                if line.strip():
                    entry = json.loads(line)
                    # Handle both 'user_id' and 'conversation_hash' keys
                    conv_hash = entry.get('conversation_hash') or entry.get('user_id')
                    if conv_hash:
                        scores[conv_hash] = {
                            'p1': entry.get('p1'),
                            'alpha': entry.get('alpha'),
                            'predictability_type': entry.get('predictability_type'),
                            'mean_exploitation_rate': entry.get('mean_exploitation_rate')
                        }
        print(f"Loaded {len(scores)} predictability scores from {filepath}")
    except FileNotFoundError:
        print(f"WARNING: {filepath} not found. Proceeding without predictability scores.")
    
    return scores

In [ ]:
def compute_priority_score(p1: float, alpha: float, categories: list) -> dict:
    """
    Compute annotation priority score based on:
    1. Unpredictability (inverse of P1 and alpha)
    2. Category relevance for belief change
    
    Higher score = higher priority for annotation
    
    Returns:
        dict with priority score, tier, and component scores
    """
    # Handle missing predictability data
    if p1 is None or alpha is None:
        unpredictability_score = 0.5  # Neutral if no data
        has_predictability = False
    else:
        # Invert P1 and alpha: low values = high unpredictability
        # P1 and alpha typically range from 0 to 1
        p1_score = 1 - min(p1, 1.0)
        alpha_score = 1 - min(alpha, 1.0)
        unpredictability_score = (p1_score * 0.5) + (alpha_score * 0.5)
        has_predictability = True
    
    # Category boost: some categories are more likely to have belief changes
    category_weights = {
        'mental_health': 0.25,       # High dissonance potential
        'life_decisions': 0.25,      # Dilemmas = cognitive conflict
        'beliefs_opinions': 0.20,    # Direct belief content
        'self_reflection': 0.20,     # Introspection often leads to change
        'persuasion': 0.20,          # Explicit belief conflict
        'challenges': 0.15,          # Problems may trigger reevaluation
        'personal_life': 0.10,       # Context for changes
    }
    
    category_boost = 0
    for cat in categories:
        category_boost += category_weights.get(cat, 0.05)
    category_boost = min(category_boost, 0.4)  # Cap at 0.4
    
    # Combined priority score
    # Weight: 60% unpredictability, 40% category relevance
    priority = (unpredictability_score * 0.6) + (category_boost * 0.4)
    priority = min(priority, 1.0)
    
    # Assign tier
    if priority >= TIER_1_THRESHOLD:
        tier = 1
        tier_label = "GOLD"
    elif priority >= TIER_2_THRESHOLD:
        tier = 2
        tier_label = "GOOD"
    else:
        tier = 3
        tier_label = "CONTROL"
    
    return {
        'priority_score': round(priority, 4),
        'tier': tier,
        'tier_label': tier_label,
        'unpredictability_score': round(unpredictability_score, 4),
        'category_boost': round(category_boost, 4),
        'has_predictability_data': has_predictability
    }

In [ ]:
# Load predictability scores
predictability_scores = load_predictability_scores(PREDICTABILITY_FILE)

# Add predictability and priority to merged data
for item in merged_data:
    conv_hash = item['conversation_hash']
    
    # Get predictability scores
    pred_data = predictability_scores.get(conv_hash, {})
    item['p1'] = pred_data.get('p1')
    item['alpha'] = pred_data.get('alpha')
    item['predictability_type'] = pred_data.get('predictability_type')
    item['mean_exploitation_rate'] = pred_data.get('mean_exploitation_rate')
    
    # Compute priority score
    priority = compute_priority_score(
        p1=item['p1'],
        alpha=item['alpha'],
        categories=item['categories']
    )
    item.update(priority)

# Sort by priority (highest first)
merged_data.sort(key=lambda x: x['priority_score'], reverse=True)

print(f"Added priority scores to {len(merged_data)} conversations")

In [ ]:
# Priority distribution
tier_counts = {1: 0, 2: 0, 3: 0}
has_pred_count = 0

for item in merged_data:
    tier_counts[item['tier']] += 1
    if item['has_predictability_data']:
        has_pred_count += 1

print("\n" + "="*60)
print("PRIORITY DISTRIBUTION")
print("="*60)
print(f"\nTier 1 (GOLD - priority >= {TIER_1_THRESHOLD}):    {tier_counts[1]:,} conversations")
print(f"Tier 2 (GOOD - priority >= {TIER_2_THRESHOLD}):    {tier_counts[2]:,} conversations")
print(f"Tier 3 (CONTROL - priority < {TIER_2_THRESHOLD}): {tier_counts[3]:,} conversations")
print(f"\nWith predictability data: {has_pred_count:,} / {len(merged_data):,}")

# Show score distribution
scores = [item['priority_score'] for item in merged_data]
print(f"\nPriority score stats:")
print(f"  Mean:   {np.mean(scores):.3f}")
print(f"  Median: {np.median(scores):.3f}")
print(f"  Min:    {np.min(scores):.3f}")
print(f"  Max:    {np.max(scores):.3f}")

In [ ]:
# Preview top priority conversations
print("\n" + "="*60)
print("TOP 10 PRIORITY CONVERSATIONS (Gold Candidates)")
print("="*60)

for i, item in enumerate(merged_data[:10]):
    print(f"\n{i+1}. Priority: {item['priority_score']:.3f} | Tier: {item['tier_label']}")
    print(f"   P1: {item['p1']}, α: {item['alpha']}, Type: {item['predictability_type']}")
    print(f"   Categories: {item['categories']}")
    print(f"   Intent: {item['intent'][:80]}...")
    print(f"   Turns: {item['num_turns']}")

In [ ]:
# Save prioritized data
with open(PRIORITIZED_FILE, 'w') as f:
    for item in merged_data:
        f.write(json.dumps(item, default=str) + '\n')

print(f"Saved {len(merged_data)} prioritized conversations to {PRIORITIZED_FILE}")
print(f"\nRecommended annotation order:")
print(f"  1. Start with Tier 1 (GOLD): {tier_counts[1]} conversations")
print(f"  2. Then Tier 2 (GOOD): {tier_counts[2]} conversations")
print(f"  3. Use Tier 3 (CONTROL) for comparison: {tier_counts[3]} conversations")

## Step 4: LLM Content Filtering

Verify that conversations actually contain personal/belief-related content.
Intent filtering is just a first pass — we need to check the actual conversation.

In [ ]:
# Content filtering prompt
CONTENT_FILTER_PROMPT = """Analyze this conversation and classify whether it contains PERSONAL or BELIEF-RELATED content.

We are looking for conversations where the user discusses:
- Personal life, relationships, family, emotions
- Mental health, anxiety, stress, challenges
- Beliefs, opinions, values, worldviews
- Life decisions, dilemmas, advice-seeking
- Self-reflection, identity, personal growth
- Conflicts, disagreements, persuasion

We are NOT interested in:
- Pure technical/coding questions
- Academic homework or exam help
- Creative writing prompts (fiction)
- Simple factual lookups
- Jailbreaking or roleplay attempts

=== CONVERSATION ===
{conversation}
=== END CONVERSATION ===

Respond in JSON format:
{{
  "is_personal_or_belief": true/false,
  "content_categories": ["mental_health", "relationships", "life_decisions", "beliefs", "self_reflection", "challenges", "other"],
  "reasoning": "brief explanation",
  "confidence": 0.0-1.0
}}

Only output valid JSON, no other text."""

In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================
# Choose your inference backend:
#   - 'huggingface': Load model locally (requires GPU)
#   - 'nebius': Use Nebius API (requires NEBIUS_API_KEY env var)
# ============================================================

INFERENCE_BACKEND = 'nebius'  # Options: 'huggingface' or 'nebius'

# HuggingFace settings (if using local model)
HF_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Nebius API settings (if using API)
NEBIUS_MODEL_NAME = "Qwen/Qwen3.5-397B-A17B-fast"
NEBIUS_BASE_URL = "https://api.tokenfactory.us-central1.nebius.com/v1/"

# NLI Model for CDR computation (always loaded locally, it's small)
NLI_MODEL_NAME = "cross-encoder/nli-deberta-v3-small"  # 44M params, fast

# Embedding model for importance weighting
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # 22M params

In [ ]:
import os
import torch

# Initialize inference backend
if INFERENCE_BACKEND == 'nebius':
    from openai import OpenAI
    
    nebius_client = OpenAI(
        base_url=NEBIUS_BASE_URL,
        api_key=os.environ.get("NEBIUS_API_KEY")
    )
    print(f"Initialized Nebius API client with model: {NEBIUS_MODEL_NAME}")
    
    # Set these to None so we don't accidentally use them
    model = None
    tokenizer = None

elif INFERENCE_BACKEND == 'huggingface':
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    print(f"Loading {HF_MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()
    print("HuggingFace model loaded!")
    
    nebius_client = None

else:
    raise ValueError(f"Unknown INFERENCE_BACKEND: {INFERENCE_BACKEND}")

In [ ]:
# Load NLI model for CDR classification (always local, small model)
from transformers import AutoModelForSequenceClassification, AutoTokenizer as NLITokenizer
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

print(f"Loading NLI model: {NLI_MODEL_NAME}...")
nli_tokenizer = NLITokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.eval()
if torch.cuda.is_available():
    nli_model = nli_model.cuda()
print("NLI model loaded!")

print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Embedding model loaded!")

In [ ]:
def format_conversation_for_prompt(conversation: list, max_chars: int = 4000) -> str:
    """
    Format conversation for LLM prompt.
    """
    formatted = []
    total_chars = 0
    
    for i, turn in enumerate(conversation):
        role = turn.get('role', 'unknown').upper()
        content = turn.get('content', '').strip()
        
        # Truncate long turns
        if len(content) > 500:
            content = content[:500] + "...[truncated]"
        
        turn_text = f"[Turn {i}] {role}: {content}"
        
        if total_chars + len(turn_text) > max_chars:
            formatted.append("...[conversation truncated]")
            break
        
        formatted.append(turn_text)
        total_chars += len(turn_text)
    
    return "\n\n".join(formatted)


def parse_llm_json_response(response: str) -> dict:
    """
    Parse JSON from LLM response, handling common formatting issues.
    """
    response = response.strip()
    
    # Remove markdown code blocks
    if response.startswith("```json"):
        response = response[7:]
    elif response.startswith("```"):
        response = response[3:]
    if response.endswith("```"):
        response = response[:-3]
    
    response = response.strip()
    
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON", "raw_response": response[:500]}

In [ ]:
# ============================================================
# UNIFIED LLM INFERENCE FUNCTION
# ============================================================
# This function handles both HuggingFace and Nebius backends
# ============================================================

def llm_generate(prompt: str, system_prompt: str = None, max_tokens: int = 1024, temperature: float = 0.1) -> str:
    """
    Generate text using the configured LLM backend.
    
    Args:
        prompt: User prompt/message
        system_prompt: Optional system prompt
        max_tokens: Maximum tokens to generate
        temperature: Sampling temperature
    
    Returns:
        Generated text string
    """
    if INFERENCE_BACKEND == 'nebius':
        messages = []
        
        if system_prompt:
            messages.append({
                "role": "system",
                "content": system_prompt
            })
        
        messages.append({
            "role": "user",
            "content": [{"type": "text", "text": prompt}]
        })
        
        response = nebius_client.chat.completions.create(
            model=NEBIUS_MODEL_NAME,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature
        )
        
        return response.choices[0].message.content
    
    elif INFERENCE_BACKEND == 'huggingface':
        messages = []
        
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        
        messages.append({"role": "user", "content": prompt})
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=True if temperature > 0 else False
            )
        
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return response
    
    else:
        raise ValueError(f"Unknown backend: {INFERENCE_BACKEND}")


# Test the inference
print("Testing LLM inference...")
test_response = llm_generate("Say 'hello' and nothing else.")
print(f"Test response: {test_response[:100]}")

In [ ]:
# ============================================================
# NLI-BASED COGNITION CLASSIFICATION
# ============================================================

@torch.inference_mode()
def classify_cognition_nli(user_belief: str, assistant_statement: str) -> dict:
    """
    Classify whether assistant's statement is dissonant, consonant, or neutral
    relative to user's belief using NLI model.
    
    Returns:
        dict with 'type' (dissonant/consonant/neutral) and 'confidence'
    """
    # NLI models expect: premise [SEP] hypothesis
    # We treat user_belief as premise, assistant_statement as hypothesis
    
    inputs = nli_tokenizer(
        user_belief, 
        assistant_statement,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    outputs = nli_model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1)[0]
    
    # DeBERTa NLI labels: 0=contradiction, 1=neutral, 2=entailment
    labels = ['contradiction', 'neutral', 'entailment']
    predicted_idx = probs.argmax().item()
    predicted_label = labels[predicted_idx]
    confidence = probs[predicted_idx].item()
    
    # Map to our terminology
    type_mapping = {
        'contradiction': 'dissonant',
        'entailment': 'consonant',
        'neutral': 'neutral'
    }
    
    return {
        'type': type_mapping[predicted_label],
        'confidence': round(confidence, 4),
        'probs': {
            'dissonant': round(probs[0].item(), 4),
            'neutral': round(probs[1].item(), 4),
            'consonant': round(probs[2].item(), 4)
        }
    }


def compute_importance_embedding(user_belief: str, assistant_statement: str) -> float:
    """
    Compute importance weight as semantic similarity between
    user's belief and assistant's statement.
    
    Returns:
        Similarity score in [0, 1] range
    """
    embeddings = embedding_model.encode([user_belief, assistant_statement])
    
    # Cosine similarity
    similarity = np.dot(embeddings[0], embeddings[1]) / (
        np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])
    )
    
    # Normalize to [0, 1] (cosine sim is in [-1, 1])
    importance = (similarity + 1) / 2
    
    return round(float(importance), 4)


# Test NLI and embedding
print("\nTesting NLI classification...")
test_nli = classify_cognition_nli(
    "I believe exercise is important for health",
    "Actually, rest is more important than exercise"
)
print(f"NLI result: {test_nli}")

print("\nTesting importance computation...")
test_importance = compute_importance_embedding(
    "I believe exercise is important for health",
    "Regular physical activity improves cardiovascular health"
)
print(f"Importance score: {test_importance}")

In [ ]:
def filter_conversation_content(conversation: list) -> dict:
    """
    Use LLM to verify conversation contains personal/belief content.
    Works with both HuggingFace and Nebius backends.
    """
    conv_text = format_conversation_for_prompt(conversation)
    prompt = CONTENT_FILTER_PROMPT.format(conversation=conv_text)
    
    response = llm_generate(prompt, max_tokens=256, temperature=0.1)
    return parse_llm_json_response(response)

In [ ]:
# Run content filtering on all conversations
print(f"Running LLM content filtering on {len(merged_data)} conversations...")

content_filtered_data = []
filter_stats_llm = {"passed": 0, "rejected": 0, "errors": 0}

for item in tqdm(merged_data, desc="Content filtering"):
    result = filter_conversation_content(item['conversation'])
    
    item['content_filter'] = result
    
    if result.get('error'):
        filter_stats_llm['errors'] += 1
        # Keep conversations with errors for manual review
        content_filtered_data.append(item)
    elif result.get('is_personal_or_belief', False):
        filter_stats_llm['passed'] += 1
        content_filtered_data.append(item)
    else:
        filter_stats_llm['rejected'] += 1

print(f"\nContent filtering results:")
print(f"  Passed:   {filter_stats_llm['passed']}")
print(f"  Rejected: {filter_stats_llm['rejected']}")
print(f"  Errors:   {filter_stats_llm['errors']}")
print(f"  Kept:     {len(content_filtered_data)} conversations")

## Step 5: LLM Belief Change Annotation

Use LLM to annotate each conversation for belief changes based on our theoretical framework:
- Cognitive Dissonance Resolution (Festinger)
- Looking-Glass Self Update (Cooley)
- Goal Shift (Cooper)
- Front-Stage Adjustment (Goffman)

In [ ]:
# Belief change annotation prompt
BELIEF_CHANGE_ANNOTATION_PROMPT = """Analyze this conversation between a user and an AI assistant for evidence of BELIEF CHANGES or PERSONA SHIFTS.

Based on psychological and sociological theories, look for:

1. **DISSONANCE_RESOLUTION** (Festinger's Cognitive Dissonance):
   - User receives information that conflicts with their prior belief
   - User UPDATES their belief to match new information
   - Phrases: "I never thought of it that way", "that makes sense", "you're right"

2. **CONSONANCE_PRESERVATION** (Festinger's Cognitive Dissonance):
   - User receives conflicting information but REJECTS it
   - User maintains original belief despite evidence
   - Phrases: "I still think...", "but that doesn't apply to...", "I disagree because..."

3. **LOOKING_GLASS_UPDATE** (Cooley's Looking-Glass Self):
   - User adjusts self-view based on assistant's feedback/judgment
   - User modifies behavior after external perspective
   - Phrases: "maybe I was wrong", "I didn't realize I was...", "you're right, I should..."

4. **GOAL_SHIFT** (Cooper's Goal-Directed Persona):
   - User's objective changes during conversation
   - Initial goal gets refined, abandoned, or replaced
   - Phrases: "actually, what I really want is...", "forget about that", "let me rephrase"

5. **FRONT_STAGE_ADJUSTMENT** (Goffman's Dramaturgy):
   - User modifies how they present themselves
   - Shift in tone, formality, or self-disclosure level
   - May reveal more personal information as trust builds

=== CONVERSATION ===
{conversation}
=== END CONVERSATION ===

Respond in JSON format:
{{
  "has_belief_change": true/false,
  "num_changes": 0,
  "changes": [
    {{
      "turn_index": N,
      "change_type": "dissonance_resolution|consonance_preservation|looking_glass_update|goal_shift|front_stage_adjustment",
      "belief_before": "user's belief/state before the change",
      "trigger": "what caused the change (quote or describe assistant's input)",
      "belief_after": "user's belief/state after the change",
      "evidence": "quote from user showing the change",
      "cdr_level": "low|medium|high"
    }}
  ],
  "overall_cdr": "low|medium|high",
  "confidence": 0.0-1.0
}}

Only output valid JSON, no other text."""

In [ ]:
def annotate_belief_changes(conversation: list) -> dict:
    """
    Use LLM to annotate belief changes in conversation.
    Works with both HuggingFace and Nebius backends.
    """
    conv_text = format_conversation_for_prompt(conversation)
    prompt = BELIEF_CHANGE_ANNOTATION_PROMPT.format(conversation=conv_text)
    
    response = llm_generate(prompt, max_tokens=1024, temperature=0.1)
    return parse_llm_json_response(response)

In [ ]:
# Run belief change annotation on content-filtered conversations
print(f"Running LLM belief change annotation on {len(content_filtered_data)} conversations...")

annotated_data = []
annotation_stats = {
    "has_change": 0,
    "no_change": 0,
    "errors": 0,
    "change_types": defaultdict(int),
    "cdr_levels": defaultdict(int)
}

for item in tqdm(content_filtered_data, desc="Annotating belief changes"):
    annotation = annotate_belief_changes(item['conversation'])
    
    item['belief_annotation'] = annotation
    annotated_data.append(item)
    
    if annotation.get('error'):
        annotation_stats['errors'] += 1
    elif annotation.get('has_belief_change', False):
        annotation_stats['has_change'] += 1
        # Count change types
        for change in annotation.get('changes', []):
            annotation_stats['change_types'][change.get('change_type', 'unknown')] += 1
        annotation_stats['cdr_levels'][annotation.get('overall_cdr', 'unknown')] += 1
    else:
        annotation_stats['no_change'] += 1

print(f"\nBelief change annotation results:")
print(f"  With changes:    {annotation_stats['has_change']}")
print(f"  Without changes: {annotation_stats['no_change']}")
print(f"  Errors:          {annotation_stats['errors']}")

print(f"\nChange types found:")
for ctype, count in sorted(annotation_stats['change_types'].items(), key=lambda x: -x[1]):
    print(f"  {ctype}: {count}")

print(f"\nCDR levels:")
for level, count in annotation_stats['cdr_levels'].items():
    print(f"  {level}: {count}")

## Step 5.5: Compute Cognitive Dissonance Ratio (CDR)

Based on Festinger's Cognitive Dissonance Theory:

```
CDR = Σ(w_d × n_d) / (Σ(w_d × n_d) + Σ(w_c × n_c))
```

Where:
- `n_d` = dissonant cognitions (assistant statements contradicting user's belief)
- `n_c` = consonant cognitions (assistant statements supporting user's belief)
- `w` = importance weight (semantic relevance to user's core claim)

CDR ranges from 0 to 1:
- **CDR ≈ 0**: Low dissonance (mostly supporting info)
- **CDR ≈ 0.5**: Balanced conflict
- **CDR ≈ 1**: High dissonance (mostly contradicting info)

We keep both:
- `cdr_llm_estimate`: LLM's subjective judgment (low/medium/high)
- `cdr_computed`: Quantitative score from formula

In [ ]:
# Prompt to extract user's initial belief (still needs LLM)
USER_BELIEF_EXTRACTION_PROMPT = """Analyze this conversation and extract the user's INITIAL belief, goal, or preference.

Look at the user's first 1-3 messages and identify:
- What they believe or think about a topic
- What goal they are trying to achieve
- What preference or opinion they hold

=== CONVERSATION ===
{conversation}
=== END CONVERSATION ===

Respond in JSON format:
{{
  "user_initial_belief": "A clear, concise statement of the user's initial belief/goal/preference",
  "belief_type": "belief|goal|preference|opinion",
  "confidence": 0.0-1.0
}}

Only output valid JSON, no other text."""

In [ ]:
def extract_user_belief(conversation: list) -> dict:
    """
    Use LLM to extract user's initial belief from conversation.
    This is the only LLM call in CDR computation.
    """
    conv_text = format_conversation_for_prompt(conversation, max_chars=2000)
    prompt = USER_BELIEF_EXTRACTION_PROMPT.format(conversation=conv_text)
    
    response = llm_generate(prompt, max_tokens=256, temperature=0.1)
    return parse_llm_json_response(response)


def compute_cdr_nli(conversation: list) -> dict:
    """
    Compute Cognitive Dissonance Ratio using:
    - LLM: Extract user's initial belief (one call)
    - NLI Model: Classify each assistant turn as dissonant/consonant/neutral
    - Embedding Model: Compute importance weight as semantic similarity
    
    Returns:
        dict with CDR analysis including computed score and per-turn cognitions
    """
    # Step 1: Extract user's initial belief using LLM
    belief_result = extract_user_belief(conversation)
    
    if belief_result.get('error'):
        return {'error': 'Failed to extract user belief', 'details': belief_result}
    
    user_belief = belief_result.get('user_initial_belief', '')
    if not user_belief:
        return {'error': 'No user belief extracted'}
    
    # Step 2: Process each assistant turn with NLI + embedding
    cognitions = []
    sum_dissonant = 0.0
    sum_consonant = 0.0
    
    for i, turn in enumerate(conversation):
        if turn.get('role', '').lower() != 'assistant':
            continue
        
        assistant_content = turn.get('content', '').strip()
        if not assistant_content or len(assistant_content) < 10:
            continue
        
        # Truncate very long responses
        if len(assistant_content) > 1000:
            assistant_content = assistant_content[:1000]
        
        # NLI classification
        nli_result = classify_cognition_nli(user_belief, assistant_content)
        cognition_type = nli_result['type']
        nli_confidence = nli_result['confidence']
        
        # Embedding-based importance
        importance = compute_importance_embedding(user_belief, assistant_content)
        
        # Weight = importance * confidence (both in [0, 1])
        weighted_score = importance * nli_confidence
        
        # Accumulate
        if cognition_type == 'dissonant':
            sum_dissonant += weighted_score
        elif cognition_type == 'consonant':
            sum_consonant += weighted_score
        # neutral doesn't contribute
        
        cognitions.append({
            'turn_index': i,
            'type': cognition_type,
            'nli_confidence': nli_confidence,
            'nli_probs': nli_result['probs'],
            'importance': importance,
            'weighted_score': round(weighted_score, 4),
            'content_preview': assistant_content[:100] + '...' if len(assistant_content) > 100 else assistant_content
        })
    
    # Step 3: Compute CDR
    total = sum_dissonant + sum_consonant
    if total > 0:
        cdr_computed = sum_dissonant / total
    else:
        cdr_computed = 0.5  # Neutral if no relevant cognitions
    
    # Interpret CDR
    if cdr_computed < 0.33:
        cdr_interpretation = 'low'
    elif cdr_computed < 0.66:
        cdr_interpretation = 'medium'
    else:
        cdr_interpretation = 'high'
    
    return {
        'user_initial_belief': user_belief,
        'belief_type': belief_result.get('belief_type'),
        'cognitions': cognitions,
        'sum_dissonant_weighted': round(sum_dissonant, 4),
        'sum_consonant_weighted': round(sum_consonant, 4),
        'cdr_computed': round(cdr_computed, 4),
        'cdr_interpretation': cdr_interpretation,
        'method': 'nli_embedding'  # Mark that this used NLI+embedding, not pure LLM
    }

In [ ]:
# Compute CDR for all annotated conversations using NLI + Embedding
print(f"Computing CDR for {len(annotated_data)} conversations...")
print("Method: NLI (classification) + Embedding (importance)")

cdr_stats = {
    'computed': [],
    'errors': 0,
    'interpretation_counts': defaultdict(int)
}

for item in tqdm(annotated_data, desc="Computing CDR"):
    cdr_result = compute_cdr_nli(item['conversation'])
    
    # Store both LLM estimate (from belief annotation) and computed CDR
    item['cdr_analysis'] = {
        'cdr_llm_estimate': item.get('belief_annotation', {}).get('overall_cdr', 'unknown'),
        'cdr_computed': cdr_result.get('cdr_computed'),
        'cdr_interpretation': cdr_result.get('cdr_interpretation'),
        'user_initial_belief': cdr_result.get('user_initial_belief'),
        'belief_type': cdr_result.get('belief_type'),
        'cognitions': cdr_result.get('cognitions', []),
        'sum_dissonant_weighted': cdr_result.get('sum_dissonant_weighted'),
        'sum_consonant_weighted': cdr_result.get('sum_consonant_weighted'),
        'method': cdr_result.get('method', 'unknown')
    }
    
    if cdr_result.get('error'):
        cdr_stats['errors'] += 1
    else:
        if cdr_result.get('cdr_computed') is not None:
            cdr_stats['computed'].append(cdr_result['cdr_computed'])
        cdr_stats['interpretation_counts'][cdr_result.get('cdr_interpretation', 'unknown')] += 1

print(f"\nCDR Computation Results:")
print(f"  Successfully computed: {len(cdr_stats['computed'])}")
print(f"  Errors: {cdr_stats['errors']}")

if cdr_stats['computed']:
    print(f"\nCDR Statistics:")
    print(f"  Mean:   {np.mean(cdr_stats['computed']):.3f}")
    print(f"  Median: {np.median(cdr_stats['computed']):.3f}")
    print(f"  Min:    {np.min(cdr_stats['computed']):.3f}")
    print(f"  Max:    {np.max(cdr_stats['computed']):.3f}")

print(f"\nCDR Interpretation Distribution:")
for interp, count in cdr_stats['interpretation_counts'].items():
    print(f"  {interp}: {count}")

In [ ]:
# Compare LLM estimate vs computed CDR
print("\n" + "="*60)
print("CDR COMPARISON: LLM Estimate vs Computed")
print("="*60)

comparison = defaultdict(lambda: defaultdict(int))

for item in annotated_data:
    llm_est = item.get('cdr_analysis', {}).get('cdr_llm_estimate', 'unknown')
    computed_interp = item.get('cdr_analysis', {}).get('cdr_interpretation', 'unknown')
    comparison[llm_est][computed_interp] += 1

print("\nConfusion Matrix (rows=LLM estimate, cols=computed):")
print(f"{'LLM \\ Computed':<15} {'low':<8} {'medium':<8} {'high':<8} {'error':<8}")
print("-" * 47)
for llm_est in ['low', 'medium', 'high', 'unknown']:
    row = comparison[llm_est]
    print(f"{llm_est:<15} {row['low']:<8} {row['medium']:<8} {row['high']:<8} {row.get('error', 0):<8}")

# Calculate agreement
agree = 0
total = 0
for llm_est in ['low', 'medium', 'high']:
    for computed in ['low', 'medium', 'high']:
        count = comparison[llm_est][computed]
        total += count
        if llm_est == computed:
            agree += count

if total > 0:
    print(f"\nAgreement rate: {agree}/{total} = {agree/total*100:.1f}%")

## Step 6: Save Annotated Dataset

Save the fully LLM-annotated dataset for experiments.

In [ ]:
# Save the fully annotated dataset
FINAL_OUTPUT_FILE = "belief_change_annotated_dataset.jsonl"

# Prepare final output format
final_data = []

for item in annotated_data:
    # Format conversation for readability
    formatted_turns = []
    for i, turn in enumerate(item['conversation']):
        formatted_turns.append({
            'turn_index': i,
            'role': turn.get('role', 'unknown'),
            'content': turn.get('content', '')
        })
    
    final_entry = {
        # Metadata
        'conversation_hash': item['conversation_hash'],
        'intent': item['intent'],
        'intent_categories': item['categories'],
        'country': item.get('country'),
        'model': item.get('model'),
        'num_turns': item['num_turns'],
        
        # Priority info
        'p1': item.get('p1'),
        'alpha': item.get('alpha'),
        'predictability_type': item.get('predictability_type'),
        'priority_score': item.get('priority_score'),
        'tier': item.get('tier'),
        'tier_label': item.get('tier_label'),
        
        # Conversation
        'turns': formatted_turns,
        
        # LLM Content Filter Result
        'content_filter': item.get('content_filter', {}),
        
        # LLM Belief Change Annotation
        'belief_annotation': item.get('belief_annotation', {}),
        
        # CDR Analysis (both LLM estimate and computed)
        'cdr_analysis': item.get('cdr_analysis', {})
    }
    
    final_data.append(final_entry)

# Sort by priority (highest first)
final_data.sort(key=lambda x: x.get('priority_score', 0), reverse=True)

# Save
with open(FINAL_OUTPUT_FILE, 'w') as f:
    for item in final_data:
        f.write(json.dumps(item, default=str) + '\n')

print(f"Saved {len(final_data)} fully annotated conversations to {FINAL_OUTPUT_FILE}")

In [ ]:
# Also save separate files for different analysis needs

# 1. Conversations WITH belief changes (for simulation experiments)
with_changes = [d for d in final_data if d['belief_annotation'].get('has_belief_change', False)]
with open('belief_change_positive.jsonl', 'w') as f:
    for item in with_changes:
        f.write(json.dumps(item, default=str) + '\n')
print(f"Saved {len(with_changes)} conversations WITH belief changes to belief_change_positive.jsonl")

# 2. Conversations WITHOUT belief changes (control group)
without_changes = [d for d in final_data if not d['belief_annotation'].get('has_belief_change', False)]
with open('belief_change_negative.jsonl', 'w') as f:
    for item in without_changes:
        f.write(json.dumps(item, default=str) + '\n')
print(f"Saved {len(without_changes)} conversations WITHOUT belief changes to belief_change_negative.jsonl")

# 3. Tier 1 (Gold) conversations only
tier1 = [d for d in final_data if d.get('tier') == 1]
with open('belief_change_tier1_gold.jsonl', 'w') as f:
    for item in tier1:
        f.write(json.dumps(item, default=str) + '\n')
print(f"Saved {len(tier1)} TIER 1 (GOLD) conversations to belief_change_tier1_gold.jsonl")

In [ ]:
# Preview a sample annotated conversation
print("\n" + "="*60)
print("SAMPLE ANNOTATED CONVERSATION")
print("="*60)

# Find a conversation with belief changes
sample = None
for item in final_data:
    if item['belief_annotation'].get('has_belief_change', False):
        sample = item
        break

if not sample:
    sample = final_data[0] if final_data else None

if sample:
    print(f"\nConversation: {sample['conversation_hash'][:20]}...")
    print(f"Intent: {sample['intent'][:80]}...")
    print(f"Priority: {sample['priority_score']} (Tier {sample['tier_label']})")
    print(f"P1: {sample['p1']}, α: {sample['alpha']}")
    
    print(f"\nContent Filter: {sample['content_filter'].get('is_personal_or_belief', 'N/A')}")
    print(f"Categories: {sample['content_filter'].get('content_categories', [])}")
    
    print(f"\nBelief Change: {sample['belief_annotation'].get('has_belief_change', 'N/A')}")
    
    # CDR Comparison
    cdr = sample.get('cdr_analysis', {})
    print(f"\nCDR Analysis:")
    print(f"  LLM Estimate:  {cdr.get('cdr_llm_estimate', 'N/A')}")
    print(f"  Computed CDR:  {cdr.get('cdr_computed', 'N/A')}")
    print(f"  Interpretation: {cdr.get('cdr_interpretation', 'N/A')}")
    print(f"  User's Initial Belief: {str(cdr.get('user_initial_belief', 'N/A'))[:60]}...")
    print(f"  Dissonant Weight: {cdr.get('sum_dissonant_weighted', 'N/A')}")
    print(f"  Consonant Weight: {cdr.get('sum_consonant_weighted', 'N/A')}")
    
    changes = sample['belief_annotation'].get('changes', [])
    if changes:
        print(f"\nBelief Changes Detected ({len(changes)}):")
        for i, change in enumerate(changes[:3]):  # Show first 3
            print(f"  {i+1}. Turn {change.get('turn_index')}: {change.get('change_type')}")
            print(f"     Before: {str(change.get('belief_before', ''))[:60]}...")
            print(f"     After:  {str(change.get('belief_after', ''))[:60]}...")

## Step 7: Summary & Next Steps

In [ ]:
print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)

print(f"\n1. KEYWORD INTENT FILTERING:")
print(f"   Total intents:        {filter_stats['total']:,}")
print(f"   Personal/belief:      {filter_stats['matched']:,} ({filter_stats['matched']/filter_stats['total']*100:.1f}%)")

print(f"\n2. CONVERSATION FETCH + PRIORITIZATION:")
print(f"   Fetched:              {len(conversations):,}")
print(f"   After min turn filter: {len(merged_data):,}")
print(f"   Tier 1 (GOLD):        {tier_counts[1]:,}")
print(f"   Tier 2 (GOOD):        {tier_counts[2]:,}")
print(f"   Tier 3 (CONTROL):     {tier_counts[3]:,}")

print(f"\n3. LLM CONTENT FILTERING:")
print(f"   Passed (personal):    {filter_stats_llm['passed']:,}")
print(f"   Rejected:             {filter_stats_llm['rejected']:,}")

print(f"\n4. LLM BELIEF CHANGE ANNOTATION:")
print(f"   With belief changes:  {annotation_stats['has_change']:,}")
print(f"   Without changes:      {annotation_stats['no_change']:,}")

print(f"\n5. CDR COMPUTATION (Festinger):")
print(f"   Successfully computed: {len(cdr_stats['computed']):,}")
if cdr_stats['computed']:
    print(f"   Mean CDR:             {np.mean(cdr_stats['computed']):.3f}")
    print(f"   CDR range:            [{np.min(cdr_stats['computed']):.2f}, {np.max(cdr_stats['computed']):.2f}]")

print(f"\n6. OUTPUT FILES:")
print(f"   - {FINAL_OUTPUT_FILE} (all annotated)")
print(f"   - belief_change_positive.jsonl ({len(with_changes)} with changes)")
print(f"   - belief_change_negative.jsonl ({len(without_changes)} without changes)")
print(f"   - belief_change_tier1_gold.jsonl ({len(tier1)} gold candidates)")

print(f"\n7. NEXT STEPS:")
print(f"   a) Run simulation experiments on belief_change_positive.jsonl")
print(f"   b) Compare LLM performance at change turns vs non-change turns")
print(f"   c) Analyze: does high CDR correlate with simulation difficulty?")
print(f"   d) Compare LLM CDR estimate vs computed CDR as predictors")
print(f"   e) Compare Tier 1 (unpredictable) vs Tier 3 (predictable) users")